# 05 - Data processing + feature selection
**Run first.** Reads the raw CSV and writes `data/processed/chicago_domestic_minimal.csv`.

Leakage rules used here: chronological split, no statistics computed on the full data, rows kept (no median filling),
leaky columns excluded, feature selection uses train + validation only (2025 test is never touched).

In [1]:
import sys, os
from pathlib import Path
# works whether Jupyter starts in the project root or inside notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))      # lets us import data_utils.py from the root
print("Project root:", ROOT)

Project root: c:\Users\aru12\OneDrive\Desktop\capstone project


## 1. Settings and imports

In [2]:
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score
from sklearn.feature_selection import mutual_info_classif

TARGET = "Domestic"                      # change to "Arrest" to build the other dataset
OTHER_LABEL = "Arrest" if TARGET == "Domestic" else "Domestic"
RAW = ROOT / "data/raw/Crimes_-_2001_to_Present_20260902.csv"
OUT = ROOT / f"data/processed/chicago_{TARGET.lower()}_minimal.csv"
SEED = 42
(ROOT / "reports").mkdir(exist_ok=True); (ROOT / "data/processed").mkdir(parents=True, exist_ok=True)

## 2. Load the raw data, parse dates, sort by time

In [3]:
df = pd.read_csv(RAW, low_memory=False)
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y %I:%M:%S %p")
df = df.sort_values("Date", kind="stable").reset_index(drop=True)
print("raw rows:", len(df))
df.head(3)

raw rows: 759290


,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,12938936,JG100768,2023-01-01 14:36:00,029XX W PETERSON AVE,5007,OTHER OFFENSE,OTHER WEAPONS VIOLATION,STREET,True,False,...,50.0,2.0,26,1155460.0,1939729.0,2023,08/19/2023 03:40:26 PM,41.990421,-87.703547,"(41.990421342, -87.703546965)"
1,12938921,JG100747,2023-01-01 14:38:00,106XX S EWING AVE,051A,ASSAULT,AGGRAVATED - HANDGUN,STREET,True,False,...,10.0,52.0,04A,1202179.0,1835113.0,2023,08/19/2023 03:40:26 PM,41.702282,-87.535284,"(41.702282173, -87.535283936)"
2,12938930,JG100871,2023-01-01 14:39:00,046XX N SPRINGFIELD AVE,1320,CRIMINAL DAMAGE,TO VEHICLE,ALLEY,False,False,...,35.0,14.0,14,1149556.0,1930546.0,2023,08/19/2023 03:40:26 PM,41.965340,-87.725503,"(41.965339687, -87.725502972)"


## 3. Leakage audit - which columns must NOT be used

In [4]:
desc_dom = df["Description"].str.contains("DOMESTIC", na=False)
print(f"'Description' contains the word DOMESTIC in {desc_dom.sum():,} rows; "
      f"Domestic rate in those rows = {df.loc[desc_dom,'Domestic'].mean():.3f}")
LEAKAGE_AUDIT = {
    "ID / Case Number": "row identifiers, no predictive meaning",
    "Updated On": "record-maintenance timestamp written AFTER the incident (future info)",
    "Block": "~35k unique street strings: identifier-like, lets models memorise places",
    "IUCR / Description / FBI Code": "finer re-coding of the crime; Description literally contains 'DOMESTIC ...' -> target leak",
    "X/Y Coordinate, Location": "duplicates of Latitude/Longitude",
    "Year": "not a usable predictor for future years",
    OTHER_LABEL: f"another outcome label - must not be a feature when predicting {TARGET}",
}
pd.Series(LEAKAGE_AUDIT, name="why excluded").to_frame()

'Description' contains the word DOMESTIC in 66,592 rows; Domestic rate in those rows = 1.000


,why excluded
ID / Case Number,"row identifiers, no predictive meaning"
Updated On,record-maintenance timestamp written AFTER the...
Block,"~35k unique street strings: identifier-like, l..."
IUCR / Description / FBI Code,finer re-coding of the crime; Description lite...
"X/Y Coordinate, Location",duplicates of Latitude/Longitude
Year,not a usable predictor for future years
Arrest,another outcome label - must not be a feature ...


## 4. Cleaning (only exact duplicates are removed; missing values are NOT median-filled)

In [5]:
before = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c not in ("ID", "Case Number", "Updated On")])
print(f"exact duplicate records removed: {before-len(df):,}  -> rows kept: {len(df):,}")

df["Hour"] = df.Date.dt.hour
df["Month"] = df.Date.dt.month
df["DayOfWeek"] = df.Date.dt.dayofweek
df["IsWeekend"] = (df.DayOfWeek >= 5).astype(int)
df["Location Description"] = df["Location Description"].fillna("UNKNOWN")
df["Primary Type"] = df["Primary Type"].fillna("UNKNOWN")
df["LatLon_Missing"] = df.Latitude.isna().astype(int)
for c in ["Ward", "Community Area"]:
    df[c] = df[c].astype("Int64")

exact duplicate records removed: 1,392  -> rows kept: 757,898


## 5. Chronological split: train 2023-Jun 2024, validation Jul-Dec 2024, test 2025

In [6]:
df["split"] = np.select([df.Date < "2024-07-01", df.Date < "2025-01-01"], ["train", "val", ], "test")
print(df.split.value_counts().to_string())
print(f"\npositive rate per split ({TARGET}):")
print(df.groupby("split")[TARGET].mean().round(4).to_string())

split
train    389196
test     236927
val      131775

positive rate per split (Domestic):
split
test     0.1907
train    0.1814
val      0.1813


## 6. Feature selection step 1: mutual information (train only)

In [7]:
CAT = ["Primary Type", "Location Description", "District", "Community Area", "Ward"]
NUM = ["Hour", "Month", "DayOfWeek", "IsWeekend", "Beat", "Latitude", "Longitude", "LatLon_Missing"]
CANDIDATES = CAT + NUM
tr, va = df[df.split == "train"], df[df.split == "val"]

Xmi = tr[CANDIDATES].copy()
for c in CAT: Xmi[c] = Xmi[c].astype(str).astype("category").cat.codes
mi = pd.Series(mutual_info_classif(Xmi.fillna(-1), tr[TARGET], discrete_features=[c in CAT for c in CANDIDATES],
                                   random_state=SEED), index=CANDIDATES).sort_values(ascending=False)
mi.round(4).to_frame("mutual_info")

,mutual_info
Primary Type,0.1340
Location Description,0.1014
Longitude,0.0855
Latitude,0.0854
Beat,0.0252
Community Area,0.0209
Ward,0.0182
District,0.0181
IsWeekend,0.0150
DayOfWeek,0.0085


## 7. Feature selection step 2: greedy forward selection scored on the validation set (takes a few minutes)

In [8]:
def make(cols):
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan)
    cc = [c for c in cols if c in CAT]
    def prep(d, fit=False):
        X = d[cols].copy().astype(float, errors="ignore")
        for c in cc: X[c] = X[c].astype(str)
        if cc:
            X[cc] = enc.fit_transform(X[cc]) if fit else enc.transform(X[cc])
        return X.astype(float)
    return prep, [c in cc for c in cols]

sub = tr.sample(150_000, random_state=SEED)       # subsample only to speed up the search
def score(cols):
    prep, mask = make(cols)
    m = HistGradientBoostingClassifier(max_iter=80, learning_rate=0.12, class_weight="balanced",
                                       categorical_features=mask if any(mask) else None, random_state=SEED)
    m.fit(prep(sub, True), sub[TARGET])
    return roc_auc_score(va[TARGET], m.predict_proba(prep(va))[:, 1])

chosen, remaining, history, best = [], CANDIDATES.copy(), [], 0.5
while remaining:
    s = {c: score(chosen + [c]) for c in remaining}
    c, auc = max(s.items(), key=lambda kv: kv[1])
    if auc - best < 0.003: break                  # stop when a new feature adds < 0.003 AUC
    chosen.append(c); remaining.remove(c); best = auc
    history.append({"step": len(chosen), "added": c, "val_auc": round(auc, 4)})
    print(f"   + {c:<22} val AUC = {auc:.4f}")
sel = pd.DataFrame(history)
sel.to_csv(ROOT / f"reports/feature_selection_{TARGET.lower()}.csv", index=False)
mi.rename("mutual_info").to_csv(ROOT / f"reports/mutual_info_{TARGET.lower()}.csv")
print("\nFINAL minimal feature set:", chosen)
sel

   + Primary Type           val AUC = 0.8457
   + Location Description   val AUC = 0.9121
   + Beat                   val AUC = 0.9177

FINAL minimal feature set: ['Primary Type', 'Location Description', 'Beat']


,step,added,val_auc
0,1,Primary Type,0.8457
1,2,Location Description,0.9121
2,3,Beat,0.9177


## 8. Save the processed dataset

In [9]:
out = df[chosen + [TARGET, "split"]].copy()
out[TARGET] = out[TARGET].astype(int)
out.to_csv(OUT, index=False)
print(f"saved {OUT}  shape={out.shape}")
out.head()

saved c:\Users\aru12\OneDrive\Desktop\capstone project\data\processed\chicago_domestic_minimal.csv  shape=(757898, 5)


,Primary Type,Location Description,Beat,Domestic,split
0,OTHER OFFENSE,STREET,2413,0,train
1,ASSAULT,STREET,432,0,train
2,CRIMINAL DAMAGE,ALLEY,1723,0,train
3,DECEPTIVE PRACTICE,UNKNOWN,1424,0,train
4,CRIMINAL DAMAGE,APARTMENT,1011,0,train
